## Step 1: Environment Setup and Dependency Imports

This section prepares the environment for the Week 06 notebook by importing required libraries, configuring paths, and authenticating with external services.

### 1. Standard Python Libraries

The notebook begins by importing commonly used Python libraries:

- `os` and `sys` for file system operations and path management
- `random` and `json` for random sampling and data handling

These utilities help manage files, datasets, and configuration settings.

### 2. Adding the Project Directory to Python Path

The code dynamically adds the **Week 6 project directory** to the Python path:

```python
_week6_dir = os.path.abspath(os.path.join(os.getcwd(), "../.."))

In [ ]:
# Step 1: Environment and data subset

# Import standard Python libraries used for file handling and randomness
import os
import sys
import random
import json

# Add the Week 6 directory to the Python path
# This allows the notebook to import modules such as `pricer`
# even when running from a different folder (community-contributions/Chrys/)
_week6_dir = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if _week6_dir not in sys.path:
    sys.path.insert(0, _week6_dir)

# Import numerical and data processing libraries
import numpy as np
import pandas as pd

# Load environment variables from the .env file
from dotenv import load_dotenv

# Hugging Face login utility
from huggingface_hub import login

# Import machine learning utilities from scikit-learn
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Import custom evaluation and item classes from the project
from pricer.evaluator import evaluate
from pricer.items import Item

# Import progress bar utility for long loops
from tqdm.notebook import tqdm

# Import display function for notebooks
# If IPython is not available, fall back to print()
try:
    from IPython.display import display
except ImportError:
    display = print


# Load environment variables from .env file
load_dotenv(override=True)

# Retrieve Hugging Face API token from environment variables
hf_token = os.environ.get("HF_TOKEN")

# If the token exists, log into Hugging Face
if hf_token:
    login(hf_token, add_to_git_credential=True)

## System Information and Dataset Preparation

This section prepares the environment by retrieving system capabilities, configuring computation resources, and loading the dataset used for training.

### 1. Adding Week 4 Utilities to the Python Path

The notebook dynamically locates the **Week 4 directory** and adds it to the Python path.

This allows the notebook to import the module:


which contains utilities for retrieving hardware information.

### 2. Retrieving System Hardware Information

The function `retrieve_system_info()` gathers information about the system, including:

- CPU details
- number of logical cores
- hardware capabilities

The information is printed as formatted JSON for readability.

If the module is not available, the code continues safely with an empty configuration.

### 3. Detecting Compute Resources

The notebook then determines which compute resources are available.

#### CPU Parallelism

The number of parallel jobs (`N_JOBS`) is calculated based on the number of logical CPU cores.  
To avoid overloading the system, the number of jobs is capped at 4.

#### GPU / MPS Detection

The code checks for hardware acceleration:

- **CUDA** → NVIDIA GPUs
- **MPS** → Apple Silicon GPU acceleration
- **CPU** → fallback if no GPU is available

This determines the device used for computation.

### 4. Batch Size Configuration

A default batch size of **32** is used.

On machines with limited RAM (for example 8GB), the batch size can be reduced to **16** to avoid out-of-memory errors.

### 5. Loading the Dataset

The dataset is loaded from **Hugging Face Hub**:


It contains three splits:

- **train** — used for training the model
- **validation** — used for model tuning
- **test** — used for final evaluation

The dataset is loaded using the custom `Item.from_hub()` method.

### 6. Creating a Smaller Training Subset

To speed up experimentation, only a subset of the training data is used.


This allows faster iteration while still providing enough data for training.

The validation and test sets remain unchanged.

### 7. Baseline Error Target

A reference baseline error is defined:


This represents the performance achieved by a previous model.

The goal for this experiment is to **achieve a lower prediction error than this baseline**.

### Summary

This step prepares the machine learning workflow by:

- retrieving system hardware information
- configuring CPU/GPU usage
- loading the dataset
- defining a manageable training subset
- setting a baseline performance target

These preparations ensure that the upcoming model training steps run efficiently.

In [ ]:
# System info: add week4 to path and get CPU/GPU capabilities

# Determine the repository root directory
_repo_root = os.path.abspath(os.path.join(_week6_dir, ".."))

# Locate the Week 4 directory
_week4 = os.path.join(_repo_root, "week4")

# If the Week 4 folder exists and is not already in the Python path,
# add it so modules inside it can be imported
if os.path.isdir(_week4) and _week4 not in sys.path:
    sys.path.insert(0, _week4)

# Attempt to import system information utility
try:
    from system_info import retrieve_system_info

    # Retrieve hardware and system capability information
    SYSTEM_INFO = retrieve_system_info()

    # Display system info in formatted JSON
    print("System info:", json.dumps(SYSTEM_INFO, indent=2))

# If the system_info module is unavailable, continue without it
except Exception as e:
    SYSTEM_INFO = {}
    print("system_info not available:", e)


# Detect hardware configuration (CPU, GPU, Apple MPS)
import torch

# Determine number of CPU cores
cores = SYSTEM_INFO.get("cpu", {}).get("cores_logical") or os.cpu_count() or 4

# Limit parallel jobs to avoid overloading the machine
N_JOBS = max(1, min(4, (cores or 4) - 1))

# Detect computation device
DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else "cpu"
)

# Define batch size for training or inference
BATCH_SIZE = 32  # reduce to 16 if out-of-memory on small machines

print(f"N_JOBS={N_JOBS}, DEVICE={DEVICE}, BATCH_SIZE={BATCH_SIZE}")


# Load dataset from Hugging Face Hub
username = "ed-donner"
dataset_name = f"{username}/items_lite"

# Load train, validation, and test splits using the custom Item class
train, val, test = Item.from_hub(dataset_name)

print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test")


# Define a smaller subset of training data to speed up experiments
TRAIN_SUBSET_SIZE = 20000

# Slice the training dataset to create a smaller training set
train_small = train[:TRAIN_SUBSET_SIZE]

print(f"Using train_small with {len(train_small):,} items for training. Val and test unchanged.")


# Reference baseline error for comparison
EXAMPLE_BASELINE_ERROR = 67.75  # Example baseline error from previous run

print(f"Target: beat example baseline error ${EXAMPLE_BASELINE_ERROR:.2f}")

## Filter Training Data, Define Local Ollama Pricing, and Create a Stratified Sample

This section prepares a cleaner and more balanced training subset for the pricing experiment.

### 1. Filtering Items by Summary Quality

The function `has_good_summary()` removes items that do not contain a useful product summary.

A summary is considered valid if:
- it exists
- it is not empty after trimming spaces
- it contains at least 20 characters

This filtering step improves data quality because the pricing model depends on the product summary text to estimate price.

After filtering, the code prints:
- how many items remain
- how many items were dropped

### 2. Local Ollama Pricing Function

The notebook then defines a simple local pricing model using **Ollama** through `litellm`.

#### `messages_for(item)`
This helper function creates the prompt sent to the local language model.  
The prompt asks the model to:

- estimate the product price
- respond with only the price
- use the product summary as input context

#### `local_ollama_pricer(item)`
This function sends the prompt to a locally running Ollama model and returns the predicted price.

The model used is:

- `llama3.2`

and the local API endpoint is:

- `http://localhost:11434`

This provides a way to test local LLM-based price prediction without relying on a cloud API.

An example evaluation call is included but commented out because local inference may be slow.  
It is recommended to test with a small sample first.

### 3. Stratified Sampling by Price Range

Instead of taking the first `N` training examples directly, the notebook creates a **stratified sample**.

The function `stratified_sample()` ensures that the smaller training set contains examples from different price ranges.

#### Price bins used
- 0 to 50
- 50 to 150
- 150 to 500
- 500 to 1000

Each item is assigned to a bin based on its price.  
The function then samples proportionally from each bin so that the smaller dataset keeps a balanced distribution.

This is useful because:
- it reduces sampling bias
- it preserves representation across cheap and expensive products
- it creates a more reliable training subset

### 4. Final Training Subset

The final smaller training set is stored in:

- `train_small`

This subset is created from:
- filtered items only
- stratified sampling
- a maximum size based on `TRAIN_SUBSET_SIZE`

The code then prints the final number of items selected.

### Summary

This step improves the dataset by:
- removing weak product summaries
- defining a local Ollama-based price estimator
- creating a balanced training subset using stratified sampling

These steps make the later modeling process more robust and efficient.

In [ ]:
# Filter the dataset to keep only items that have a usable summary
def has_good_summary(item):
    # Get the item summary safely, remove extra spaces,
    # and ensure it has at least 20 characters
    s = (item.summary or "").strip()
    return len(s) >= 20

# Apply the filter to the full training dataset
train_filtered = [x for x in train if has_good_summary(x)]

# Print how many items remain after filtering
print(f"After filtering: {len(train_filtered):,} items (dropped {len(train) - len(train_filtered):,})")


# Import LiteLLM completion utility for calling a local Ollama model
from litellm import completion

# Define the local Ollama model and base URL
OLLAMA_MODEL = "llama3.2"  # can also use other local models like qwen2.5:7b
OLLAMA_BASE = "http://localhost:11434"


# Create the prompt message for one product item
def messages_for(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": msg}]


# Call the local Ollama model to estimate the product price
def local_ollama_pricer(item):
    r = completion(
        model=f"ollama/{OLLAMA_MODEL}",
        messages=messages_for(item),
        api_base=OLLAMA_BASE,
    )
    return r.choices[0].message.content


# Example evaluation call
# Uncomment to test the pricing model on a small sample first,
# because local inference can be slow
# random.seed(42)
# evaluate(local_ollama_pricer, test, size=50)


# Create a stratified sample so that the smaller dataset
# keeps a balanced distribution across price ranges
def stratified_sample(items, n, bins=(0, 50, 150, 500, 1000), random_state=42):
    rng = np.random.default_rng(random_state)

    # Group items by price bin
    by_bin = {}
    for item in items:
        p = item.price
        b = next(
            (i for i in range(len(bins) - 1) if bins[i] <= p < bins[i + 1]),
            len(bins) - 2
        )
        by_bin.setdefault(b, []).append(item)

    # Sample proportionally from each bin
    out = []
    for b in sorted(by_bin.keys()):
        pool = by_bin[b]
        k = max(1, round(n * len(pool) / len(items)))
        k = min(k, len(pool))
        idx = rng.choice(len(pool), size=k, replace=False)
        out.extend([pool[i] for i in idx])

    # Shuffle the combined sample and keep only n items
    rng.shuffle(out)
    return out[:n]


# Build a smaller, stratified training subset
train_small = stratified_sample(train_filtered, min(TRAIN_SUBSET_SIZE, len(train_filtered)))

# Print final subset size
print(f"train_small (stratified): {len(train_small):,} items")

## Feature Engineering, Traditional ML Models, and Evaluation

This section builds a traditional machine learning pipeline for product price prediction using both **text features** and **numeric features**.

### 1. Manual Feature Engineering

The function `get_features(item)` extracts simple structured features from each product:

- **weight** → product weight if available
- **weight_unknown** → indicates whether weight is missing or zero
- **text_length** → length of the product summary

These features provide additional signals beyond the raw text description.

The function `list_to_dataframe(items)` converts a list of product items into a pandas DataFrame containing:
- engineered features
- target price column

### 2. TF-IDF Text Features

The notebook uses **TF-IDF (Term Frequency–Inverse Document Frequency)** to convert product summaries into numerical vectors.

Configuration:
- maximum of **3000 features**
- English stop words removed

The TF-IDF vectorizer is fitted on `train_small` and then applied to:
- validation set
- test set

This transforms each summary into a sparse vector representation that captures important words and phrases.

### 3. Combining Text and Numeric Features

The text-based TF-IDF features are combined with the manual numeric features:

- weight
- weight_unknown
- text_length

This creates richer feature matrices for:
- training
- validation
- testing

The target variable is the product price.

### 4. Linear Regression Baseline

A **Linear Regression** model is trained as a simple baseline.

Steps:
- fit the model on the combined training features
- predict prices on the test set
- clip negative predictions to zero

Two evaluation metrics are printed:
- **Mean Squared Error (MSE)** — lower is better
- **R² score** — closer to 1 is better

This provides a basic performance benchmark.

### 5. XGBoost or RandomForest

The code then attempts to use a stronger tree-based model.

#### XGBoost
If `USE_XGBOOST = True`, the notebook tries to train an `XGBRegressor`.

However, XGBoost is disabled by default because it can fail in some environments due to missing dependencies such as `libomp`.

#### RandomForest fallback
If XGBoost is unavailable, the notebook automatically falls back to **RandomForestRegressor**.

This ensures the workflow still runs even if XGBoost cannot be used.

### 6. Pricer Functions

Two prediction wrapper functions are created:

- `linear_regression_pricer(item)`
- `xgboost_pricer(item)`

Each function:
1. converts the product summary into TF-IDF features
2. extracts numeric features
3. combines both into one feature vector
4. predicts the price
5. ensures the prediction is non-negative

These wrapper functions match the evaluator format expected by the `evaluate()` function.

### 7. Model Evaluation

Finally, both traditional ML models are evaluated on the test set:

- Linear Regression
- XGBoost or RandomForest

This allows direct comparison of model performance.

### Summary

This section demonstrates a complete traditional ML approach for price prediction using:

- manual feature engineering
- TF-IDF text features
- Linear Regression baseline
- XGBoost / RandomForest tree models
- standardized evaluation functions

It serves as an important benchmark before comparing against more advanced LLM-based pricing approaches.

In [ ]:
# Extract structured features from one item
def get_features(item):
    return {
        # Use product weight if available, otherwise default to 0
        "weight": item.weight if item.weight is not None else 0.0,

        # Binary feature indicating whether weight is missing or zero
        "weight_unknown": 1 if (item.weight is None or item.weight == 0) else 0,

        # Length of the product summary text
        "text_length": len(item.summary or ""),
    }


# Convert a list of items into a pandas DataFrame of features
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)

    # Add the target variable: product price
    df["price"] = [item.price for item in items]
    return df


# Build TF-IDF text features using only the smaller training subset
np.random.seed(42)
documents_train = [item.summary or "" for item in train_small]

# Convert summaries into TF-IDF vectors
tfidf = TfidfVectorizer(max_features=3000, stop_words="english")
X_tfidf_train = tfidf.fit_transform(documents_train)

# Transform validation and test summaries using the same fitted TF-IDF model
X_tfidf_val = tfidf.transform([item.summary or "" for item in val])
X_tfidf_test = tfidf.transform([item.summary or "" for item in test])

print(f"TF-IDF shape: {X_tfidf_train.shape}")


# Combine TF-IDF text features with manually engineered numeric features
from scipy.sparse import hstack

train_df = list_to_dataframe(train_small)
val_df = list_to_dataframe(val)
test_df = list_to_dataframe(test)

# Select numeric columns to combine with TF-IDF features
num_cols = ["weight", "weight_unknown", "text_length"]

# Create final feature matrices
X_train_combined = hstack([X_tfidf_train, train_df[num_cols].values])
X_val_combined = hstack([X_tfidf_val, val_df[num_cols].values])
X_test_combined = hstack([X_tfidf_test, test_df[num_cols].values])

# Create target arrays
y_train = train_df["price"].values
y_val = val_df["price"].values
y_test = test_df["price"].values


# Train a simple Linear Regression baseline model
lr_model = LinearRegression()
lr_model.fit(X_train_combined, y_train)

# Predict prices on the test set and clip negative predictions to zero
y_pred_lr = np.maximum(0, lr_model.predict(X_test_combined))

# Print baseline evaluation metrics
print(f"Linear Regression MSE: {mean_squared_error(y_test, y_pred_lr):.2f}, R²: {r2_score(y_test, y_pred_lr):.4f}")


# Try XGBoost, otherwise fall back to RandomForest
# XGBoost is disabled by default here because it may fail if dependencies are missing
USE_XGBOOST = False
use_xgb = False

if USE_XGBOOST:
    try:
        import xgboost as xgb

        xgb_model = xgb.XGBRegressor(
            n_estimators=400,
            max_depth=7,
            learning_rate=0.07,
            n_jobs=N_JOBS,
            random_state=42
        )

        xgb_model.fit(
            X_train_combined,
            y_train,
            eval_set=[(X_val_combined, y_val)],
            verbose=False
        )
        use_xgb = True

    except Exception:
        pass

# If XGBoost is not used, train a RandomForest regressor instead
if not use_xgb:
    from sklearn.ensemble import RandomForestRegressor

    xgb_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        n_jobs=N_JOBS,
        random_state=42
    )
    xgb_model.fit(X_train_combined, y_train)

print("XGBoost" if use_xgb else "RandomForest", "fitted.")


# Create a pricer function for the trained Linear Regression model
# The evaluator expects a function that takes one item and returns a predicted price
def linear_regression_pricer(item):
    txt = tfidf.transform([item.summary or ""])
    num = np.array([get_features(item)[c] for c in num_cols]).reshape(1, -1)
    X = hstack([txt, num])
    return max(0.0, float(lr_model.predict(X)[0]))


# Create a pricer function for the XGBoost or RandomForest model
def xgboost_pricer(item):
    txt = tfidf.transform([item.summary or ""])
    num = np.array([get_features(item)[c] for c in num_cols]).reshape(1, -1)
    X = hstack([txt, num])
    return max(0.0, float(xgb_model.predict(X)[0]))


# Evaluate the traditional machine learning models using the provided evaluator
random.seed(42)

print("Linear Regression:")
evaluate(linear_regression_pricer, test)

print("\nXGBoost/RandomForest:")
evaluate(xgboost_pricer, test)

## Neural Network Price Predictor (MLP)

This section builds a **deep learning model** for product price prediction using PyTorch.

Unlike the earlier traditional ML models, this model uses a **Multi-Layer Perceptron (MLP)** to learn more complex relationships between product descriptions and prices.

---

## 1. Neural Network Architecture

The class `MLPPricer` defines a feedforward neural network.

Architecture:

Input Layer  
→ Hidden Layer (256 neurons)  
→ Hidden Layer (128 neurons)  
→ Hidden Layer (64 neurons)  
→ Output Layer (1 value)

Each hidden layer includes:

- **Linear layer** — learns weighted relationships
- **ReLU activation** — introduces non-linearity
- **Dropout** — prevents overfitting

The final layer outputs a **single value representing the predicted price**.

---

## 2. Preparing Training Data

The feature matrices used earlier were stored as **sparse matrices**.

PyTorch requires **dense tensors**, so they are converted using:


Target values (prices) are converted into tensors and reshaped so that each example has one target value.

---

## 3. DataLoader for Batch Training

The dataset is wrapped in a `TensorDataset` and passed to a `DataLoader`.

This enables:

- mini-batch training
- shuffled batches
- efficient iteration during training

Batch size is controlled by the earlier `BATCH_SIZE` parameter.

---

## 4. Model Training

The model is trained using:

- **Optimizer:** Adam
- **Loss Function:** Mean Squared Error (MSE)

Training runs for several epochs.

For each epoch:

1. Model switches to **training mode**
2. Batches are processed
3. Gradients are computed using backpropagation
4. Weights are updated

After each epoch, validation loss is calculated to monitor performance.

---

## 5. Neural Network Pricing Function

The function `neural_pricer(item)` allows the trained neural network to be used with the project evaluator.

For each item:

1. Convert summary text into TF-IDF features
2. Extract numeric features
3. Combine both feature sets
4. Convert to a PyTorch tensor
5. Run the neural network prediction
6. Ensure predicted price is non-negative

This wrapper function matches the expected evaluator interface.

---

## 6. Model Evaluation

Finally, the model is evaluated on the **test dataset** using the provided `evaluate()` function.

This produces performance metrics that can be compared with earlier models such as:

- Linear Regression
- RandomForest / XGBoost
- Local LLM-based pricing

---

## Summary

This step introduces a **neural network regression model** that:

- combines text features and numeric features
- learns nonlinear relationships
- uses mini-batch training
- supports GPU acceleration if available

It provides a more flexible model that may outperform traditional regression approaches for complex pricing patterns.


In [ ]:
# Import utilities for batching and datasets in PyTorch
from torch.utils.data import DataLoader, TensorDataset


# Define a neural network model for price prediction
class MLPPricer(torch.nn.Module):

    def __init__(self, input_size, hidden=(256, 128, 64), dropout=0.25):
        super().__init__()

        layers = []
        prev = input_size

        # Create multiple hidden layers
        for h in hidden:
            layers.extend([
                torch.nn.Linear(prev, h),  # Fully connected layer
                torch.nn.ReLU(),           # Non-linear activation
                torch.nn.Dropout(dropout)  # Dropout to reduce overfitting
            ])
            prev = h

        # Final output layer (predict a single price value)
        layers.append(torch.nn.Linear(prev, 1))

        # Combine all layers into a sequential network
        self.net = torch.nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# Convert sparse feature matrices into dense tensors for PyTorch
X_train_dense = torch.FloatTensor(X_train_combined.toarray())
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)

X_val_dense = torch.FloatTensor(X_val_combined.toarray())
y_val_t = torch.FloatTensor(y_val).unsqueeze(1)


# Create dataset and data loader for batch training
train_ds = TensorDataset(X_train_dense, y_train_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)


# Determine input feature size
input_size = X_train_dense.shape[1]

# Initialize neural network model and move it to the compute device
nn_model = MLPPricer(input_size).to(DEVICE)

# Adam optimizer for training
optimizer = torch.optim.Adam(nn_model.parameters(), lr=1e-3)

# Mean Squared Error loss for regression
loss_fn = torch.nn.MSELoss()

print(f"MLP parameters: {sum(p.numel() for p in nn_model.parameters()):,}")


# Train the neural network
# Run for a few epochs with validation monitoring
EPOCHS = 5
best_val = float("inf")

for epoch in range(EPOCHS):

    # Training phase
    nn_model.train()
    for bx, by in train_loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)

        optimizer.zero_grad()

        # Forward pass
        loss = loss_fn(nn_model(bx), by)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

    # Validation phase
    nn_model.eval()
    with torch.no_grad():
        val_loss = loss_fn(
            nn_model(X_val_dense.to(DEVICE)),
            y_val_t.to(DEVICE)
        ).item()

    if val_loss < best_val:
        best_val = val_loss

    print(f"Epoch {epoch+1}/{EPOCHS} Val MSE: {val_loss:.2f}")


# Define a pricing function using the trained neural network
def neural_pricer(item):

    nn_model.eval()

    with torch.no_grad():

        # Convert item summary to TF-IDF features
        txt = tfidf.transform([item.summary or ""])

        # Extract numeric features
        num = np.array([get_features(item)[c] for c in num_cols]).reshape(1, -1)

        # Combine text and numeric features
        X = hstack([txt, num]).toarray()

        # Convert to tensor
        x = torch.FloatTensor(X).to(DEVICE)

        # Predict price
        out = nn_model(x)[0].item()

    return max(0.0, out)


# Evaluate the neural network model
random.seed(42)

print("Neural network (MLP):")
evaluate(neural_pricer, test)

## Evaluate and Compare All Pricing Models

This section computes evaluation metrics for each pricing model and summarizes their performance in a comparison table.

### 1. Custom Metric Function

The function `compute_metrics()` evaluates a pricing model on a fixed number of test items.

It uses the project’s `Tester` utility to run predictions one item at a time and collect:

- predicted prices
- true prices
- individual prediction errors

From these results, it computes:

- **Average Error** — the average absolute error in dollars
- **Mean Squared Error (MSE)** — gives more weight to large mistakes
- **R² score** — shows how well the predictions explain price variance

The R² value is converted to a percentage for readability.

### 2. Evaluation Size

The number of test examples is limited to:

- **200 items maximum**

This keeps evaluation comparable to the default evaluator setting while keeping runtime manageable.

### 3. Model Comparison

The notebook evaluates three models:

- **Linear Regression**
- **XGBoost / RandomForest**
- **Neural MLP**

For each model, the code:
1. computes the metrics
2. stores them in a results list
3. prints a short performance summary

### 4. Baseline Comparison

The results table also includes an external reference model:

- **Example (fine-tuned nano 20k)**

This provides a benchmark error value:

- `EXAMPLE_BASELINE_ERROR`

Each model is then marked as:

- **Yes** if it beats the baseline
- **No** otherwise

### 5. Summary Table

All results are stored in a pandas DataFrame and displayed with the following columns:

- model name
- number of training examples
- average error
- mean squared error
- R² score
- whether it beats the baseline

This makes it easy to compare model performance side by side.

### 6. Best Model Selection

Finally, the notebook selects the model with the **lowest average error** and prints it as the best-performing model.

### Summary

This step provides a unified performance comparison across all developed pricing approaches.

It helps answer key questions such as:

- Which model performs best on the test set?
- Does the custom model beat the example baseline?
- How do traditional ML and neural models compare?

This is the final evaluation stage of the Week 06 pricing workflow.

In [ ]:
# Compute evaluation metrics for any pricing function
# Uses the same evaluation size as the default evaluator (200 items)
def compute_metrics(pricer_fn, data, size=200):

    # Import the Tester utility from the project
    from pricer.evaluator import Tester

    # Create a tester object
    t = Tester(pricer_fn, data, size=size, workers=5)

    # Run evaluation on each datapoint
    for i in range(size):
        title, guess, truth, error, color = t.run_datapoint(i)

        # Store results for later metric calculation
        t.titles.append(title)
        t.guesses.append(guess)
        t.truths.append(truth)
        t.errors.append(error)
        t.colors.append(color)

    # Compute average absolute error
    avg_err = sum(t.errors) / len(t.errors)

    # Compute Mean Squared Error
    mse = mean_squared_error(t.truths, t.guesses)

    # Compute R² score and convert to percentage
    r2 = r2_score(t.truths, t.guesses) * 100

    return {"error": avg_err, "mse": mse, "r2": r2}


# Define evaluation size, capped at 200 test samples
EVAL_SIZE = min(200, len(test))

# Store model comparison results
results = []

# Evaluate each pricing model
for name, fn in [
    ("Linear Regression", linear_regression_pricer),
    ("XGBoost/RF", xgboost_pricer),
    ("Neural MLP", neural_pricer),
]:
    m = compute_metrics(fn, test, size=EVAL_SIZE)

    # Store metrics in results list
    results.append({"model": name, "train_n": len(train_small), **m})

    # Print quick summary for each model
    print(f"{name}: Error ${m['error']:.2f}, MSE={m['mse']:.0f}, r²={m['r2']:.1f}%")


# Add the example baseline model for comparison
results.append({
    "model": "Example (fine-tuned nano 20k)",
    "train_n": 20_000,
    "error": EXAMPLE_BASELINE_ERROR,
    "mse": None,
    "r2": None,
})

# Convert results into a DataFrame
summary_df = pd.DataFrame(results)

# Check whether each model beats the example baseline
summary_df["beat_baseline"] = summary_df["error"].apply(
    lambda e: "Yes" if e < EXAMPLE_BASELINE_ERROR else "No"
)

# Display the summary table
display(summary_df[["model", "train_n", "error", "mse", "r2", "beat_baseline"]])

# Find the model with the lowest average error
best = summary_df.loc[summary_df["error"].idxmin()]

print(f"\nBest model: {best['model']} (Error ${best['error']:.2f})")